In [33]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [34]:
big_path = "Data/big_data.csv"
smaller_path = "Data/smaller_data.csv"

df = pd.read_csv(smaller_path)

df['label'] = df['label'].map({0: 'real', 1: 'fake'})

# 1. Fill any NaNs that appeared during the heavy cleaning
df['text'] = df['text'].fillna("")

# 2. (Optional but recommended) Remove rows that became empty strings 
# after masking/cleaning so they don't confuse the model
df = df[df['text'].str.strip() != ""]

# Now proceed to your split
# X_train, X_test, y_train, y_test = train_test_split(df['text'], df['label'], ...)

X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42
)

In [35]:
custom_stops = list(ENGLISH_STOP_WORDS) + [
    'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday',
    'text', 'related', 'coverage', 'told', 'said', 'euros'
]

vectorizer = TfidfVectorizer(stop_words=custom_stops, max_df=0.7, min_df=5,ngram_range=(1,2))

model = make_pipeline(vectorizer, LogisticRegression())
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print(classification_report(y_test, y_pred=predictions))
score = model.score(X_test, y_test)
print(f"Model score: {score}")

              precision    recall  f1-score   support

        fake       0.87      0.81      0.84      1521
        real       0.81      0.88      0.84      1479

    accuracy                           0.84      3000
   macro avg       0.84      0.84      0.84      3000
weighted avg       0.84      0.84      0.84      3000

Model score: 0.8406666666666667


In [36]:
def predict_article(text):
    return model.predict([text])[0]

real_text = "Popular streamer Clavicular framemogged by ASU frat leader."
prediction = predict_article(real_text)

print(f"The model predicts the real text as: {prediction}")

fake_text = "Scientists discover that the moon is made out of cheese."
prediction = predict_article(fake_text)

print(f"The model predicts the fake text as: {prediction}")

The model predicts the real text as: real
The model predicts the fake text as: real


In [37]:
import pandas as pd

def print_top_features(vectorizer, model, n_top=20):
    features = vectorizer.get_feature_names_out()
    coefs = model.coef_[0]
    
    df = pd.DataFrame({'word': features, 'weight': coefs})
    
    print("--- Top Predictors for FAKE (Class 1) ---")
    print(df.sort_values(by='weight', ascending=False).head(n_top).to_string(index=False))
    
    print("\n--- Top Predictors for REAL (Class 0) ---")
    print(df.sort_values(by='weight', ascending=True).head(n_top).to_string(index=False))

# Example usage assuming your fitted models are named 'tfidf' and 'log_model'
# 1. Extract the fitted vectorizer and model from your pipeline
fitted_vectorizer = model[0]
fitted_log_reg = model[1]

# 2. Call the function using those extracted pieces
print_top_features(fitted_vectorizer, fitted_log_reg)

--- Top Predictors for FAKE (Class 1) ---
          word   weight
       persons 3.963340
          orgs 3.339868
       matters 2.599345
source company 2.531726
          gpes 2.481316
        source 2.154977
         sales 1.987041
       gpe org 1.975695
        sports 1.962474
       million 1.942069
          week 1.924107
         early 1.857276
          norp 1.854323
           gpe 1.845555
       details 1.794904
    previously 1.792060
      analysts 1.763802
      earnings 1.726319
       earlier 1.637165
     reporters 1.634873

--- Top Predictors for REAL (Class 0) ---
      word    weight
     going -4.268004
      able -3.983136
 statement -3.814189
     think -3.617249
 according -3.591379
   org org -3.549957
     image -3.494898
   gpe gpe -3.452814
  continue -3.331331
  end year -2.992525
    number -2.827223
      know -2.822983
      time -2.727611
    ensure -2.702814
claim says -2.687311
      just -2.656078
      dont -2.643676
    people -2.612093
       end -